# Simple PaCMAP fit + transform

Minimal landing example: fit PaCMAP on a reference cohort, transform new samples into that
space, plot the reference landscape vs. the landed points, and save the landed
coordinates to CSV.

## Install dependencies

In [ ]:
%pip install -q pacmap pandas numpy matplotlib

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pacmap

## Inputs

In [12]:
# CSV of the reference cohort: rows = samples, columns = genes, first column = sample ID.
# Must already be batch-corrected and VST-normalized.
# First column/index must be gene names. Other columns are sample IDs
TRAINING_VST_COUNTS = r"data\VST_counts.txt"

# CSV of the new sample(s) to land, same gene set / normalization as training data. No batch correction.
# First column/index must be gene names. Other columns are sample IDs
NEW_VST_COUNTS_TO_LAND = r"data\pdx_43_small_cell_no_hg38_vst_protein_coding_genes.txt"

# Path to write the landed coordinates.
OUTPUT_CSV_PATH = "pacmap_landed_coordinates.csv"

## Load data

In [49]:
X_train = pd.read_csv(TRAINING_VST_COUNTS, sep=DELIMITER, index_col=0)
X_test = pd.read_csv(NEW_VST_COUNTS_TO_LAND, sep=DELIMITER, index_col=0)

X_test_sample_ids = X_test.columns

assert X_train.index.equals(X_test.index), "Genes do not match between training and new data. Check that both datasets have the same genes in the same order."

X_train = X_train.T.to_numpy(dtype=np.float32)
X_test = X_test.T.to_numpy(dtype=np.float32)

train_samples, train_features = X_train.shape
test_samples, test_features = X_test.shape

print(f"Training data: {train_samples} samples, {train_features} features")
print(f"Landing data: {test_samples} samples, {test_features} features")

Training data: 1824 samples, 19055 features
Landing data: 42 samples, 19055 features


## Fit PaCMAP on the reference cohort

In [36]:
reducer = pacmap.PaCMAP(
    n_components=2,     # output dimensionality (2D landscape)
    n_neighbors=30,      # number of neighbors considered per point; None = auto-scaled by sample size
    MN_ratio=0.25,        # ratio of mid-near pairs to neighbor pairs; controls global structure
    FP_ratio=3.0,        # ratio of further pairs to neighbor pairs; controls separation between clusters
    random_state=42,      # seed for reproducibility
)

# fit_transform embeds the training data and stores what's needed for later .transform() calls
# this is the "reference landscape"
train_embedding = reducer.fit_transform(X_train)

train_embedding.shape
print(f"Training embedding shape: {train_embedding.shape}")

Training embedding shape: (1824, 2)


## Transform new samples into the fitted landscape

In [40]:
# basis=X_train tells PaCMAP to place new points relative to the original training data
landed_coordinates = reducer.transform(X_test, basis=X_train)

landed_coordinates.shape

(42, 2)